## Without web search

In [35]:
%run langchain_common.py

In [36]:
agent = create_agent(
    model=llm_noreason
)

In [37]:
agent.invoke(
    {"messages": [HumanMessage(content="who is current mayor of lahore?")]}
)['messages'][-1].content

'The current Mayor of Lahore is **Khalid Maqbool Siddiqui**.\n\nHe assumed office on **August 1, 2022**, representing the Pakistan Muslim League-Nawaz (PML-N). He was elected following the local government elections held in July 2022, succeeding the caretaker mayor, Muhammad Iqbal.'

In [38]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)
pp(response['messages'])

[
    HumanMessage(content='How up to date is your training knowledge?', additional_kwargs={}, response_metadata={}, id='b40c88a5-6047-4b01-be20-e5fcfac0a151'),
    AIMessage(content="My training data cutoff date is **2026**. This means my knowledge and understanding of events, facts, and developments are current up to that point. For information beyond this date, I may not have accurate or complete details. If you have specific questions, I'll do my best to provide relevant and up-to-date answers based on my training!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 21, 'total_tokens': 93, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_9768734f-53f8-445c-b598-33ec00df2143', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ee094-1ae3-7b43-bc11-dd8147664e74-0', tool_calls=[], invalid_too

## Add web search tool

In [39]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_KEY"))

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

resp = web_search.invoke("Who is the current mayor of Lahore?")
pp(resp)

{
    'answer': None,
    'follow_up_questions': None,
    'images': [],
    'query': 'Who is the current mayor of Lahore?',
    'request_id': '5e26950b-9ca1-40c5-9590-52cf308cb0a6',
    'response_time': 0.88,
    'results': [
        {
            'content': 'Mayors of Lahore ; Mian Amir Mehmood, 2005, 2009 ; Administrator system implemented 2010–2016 ; Mubashar Javed, 2016, 2021 ; Administrator system implemented',
            'raw_content': None,
            'score': 0.8585411,
            'title': 'Mayor of Lahore - Wikipedia',
            'url': 'https://en.wikipedia.org/wiki/Mayor_of_Lahore',
        },
        {
            'content': 'LAHORE: Retired Col Mubashir Javed of the PML-N assumed the charge of mayor office here on Saturday and the Punjab government formally',
            'raw_content': None,
            'score': 0.8377398,
            'title': 'Lahore mayor assumes charge of office - Pakistan - DAWN.COM',
            'url': 'https://www.dawn.com/news/1653640',
       

In [40]:
agent = create_agent(
    llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of Lahore?")

response = agent.invoke(
    {"messages": [question]}
)

In [41]:
pp(response['messages'])

[
    HumanMessage(content='Who is the current mayor of Lahore?', additional_kwargs={}, response_metadata={}, id='73b2d4cc-2b0d-44f8-8351-66f173794e3a'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 280, 'total_tokens': 357, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_5da0f3ad-b765-4eae-afb4-b31a236f663d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ee094-25e1-70d0-b280-a862d66b6cc4-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of Lahore 2024 2025'}, 'id': 'call_55bde9bb-4960-42e4-9ba6-5d693e00fc57', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 280, 'output_tokens': 77, 'total_tokens': 357, 'input_token_details': {}, 'output_token_details': {}}),
    ToolMessage(content='{"query": "curre

In [42]:
pp(response['messages'][-1].content)

[
    {'summary': [{'text': '<think>', 'type': 'summary_text'}], 'type': 'reasoning'},
    {
        'text': 'As of 2024, the position of the Mayor of Lahore is currently **vacant** or held by an **Administrator**.\n\nThe last elected mayor was **Colonel (Retired) Mubashir Javed** of the PML-N, who assumed office in December 2016. However, the local government system in Punjab was dissolved in January 2022, and the office of the mayor was suspended. Since then, the city has been administered by a government-appointed **Administrator** (often referred to as the District Officer or Administrator Lahore) rather than an elected mayor.\n\nThere have been ongoing discussions and legal battles regarding the restoration of local governments in Punjab, but as of now, no new mayor has been elected to replace Mubashir Javed.',
        'type': 'text',
    },
]


In [43]:
pp(response["messages"][1].tool_calls)

[
    {
        'args': {'query': 'current mayor of Lahore 2024 2025'},
        'id': 'call_55bde9bb-4960-42e4-9ba6-5d693e00fc57',
        'name': 'web_search',
        'type': 'tool_call',
    },
]


In [46]:
    question = HumanMessage(content="How is the weather today in Lahore?")

    response = agent.invoke(
        {"messages": [question]}
    )
    pp(response['messages'][-1].content)


[
    {
        'summary': [
            {
                'text': "The user is asking about the current weather in Lahore. Based on the search results, I can see there's some conflicting information about the date (some results show 2026-06-19, which seems to be a future date in the search results, likely a data issue). However, I can extract the current weather information from the first result which appears to be from weatherapi.com.\n\nFrom the first search result:\n- Location: Lahore, Punjab, Pakistan\n- Current temperature: 33.0°C (91.4°F)\n- Condition: Clear\n- Wind: 6.5 km/h from the West\n- Humidity: 47%\n- Feels like: 36.1°C (96.9°F)\n- No precipitation expected\n\nLet me provide this information to the user in a clear and helpful way.",
                'type': 'summary_text',
            },
        ],
        'type': 'reasoning',
    },
    {
        'text': "Based on the latest weather data, here's the current weather in Lahore:\n\n**Current Conditions:**\n- **Temperature:*